In [1]:
import pandas as pd
import numpy as np
import re

# loading the cleaned dataset
df = pd.read_csv("../data/youtube_india_clean.csv")
# this is the main india youtube cleaned dataset

# making a fresh copy so original df stays safe
df_model = df.copy()

# keeping only columns needed for pre-upload regression
df_model = df_model[
    [
        "video_title",
        "video_category_id",
        "video_duration",
        "video_definition",
        "video_view_count",
        "channel_subscriber_count",
        "channel_video_count",
        "channel_have_hidden_subscribers",
        "video_tags"
    ]
]
# removing unnecessary columns and keeping only useful pre-upload features

# filling missing target values
df_model["video_view_count"] = df_model["video_view_count"].fillna(
    df_model["video_view_count"].median()
)
# important because target cannot have NaN

# filling missing subscriber count
df_model["channel_subscriber_count"] = df_model["channel_subscriber_count"].fillna(
    df_model["channel_subscriber_count"].median()
)
# using median because subscriber data can be skewed

# filling missing category with most common value
df_model["video_category_id"] = df_model["video_category_id"].fillna(
    df_model["video_category_id"].mode()[0]
)

# filling missing tags with empty string
df_model["video_tags"] = df_model["video_tags"].fillna("")
# needed because later we combine title + tags

# function to convert youtube duration into seconds
def convert_duration_to_seconds(duration):
    hours = 0
    minutes = 0
    seconds = 0

    h = re.search(r"(\d+)H", str(duration))
    m = re.search(r"(\d+)M", str(duration))
    s = re.search(r"(\d+)S", str(duration))

    if h:
        hours = int(h.group(1))
    if m:
        minutes = int(m.group(1))
    if s:
        seconds = int(s.group(1))

    return hours * 3600 + minutes * 60 + seconds

# creating numeric duration column
df_model["video_duration_seconds"] = df_model["video_duration"].apply(convert_duration_to_seconds)

# dropping original duration text column
df_model = df_model.drop(columns=["video_duration"])

# creating log target for stable regression
df_model["log_views"] = np.log1p(df_model["video_view_count"])
# log1p helps because view counts are very skewed

print("Prepared dataset shape:", df_model.shape)
print("\nColumns:")
print(df_model.columns)

df_model.head()

Prepared dataset shape: (302786, 10)

Columns:
Index(['video_title', 'video_category_id', 'video_definition',
       'video_view_count', 'channel_subscriber_count', 'channel_video_count',
       'channel_have_hidden_subscribers', 'video_tags',
       'video_duration_seconds', 'log_views'],
      dtype='object')


,video_title,video_category_id,video_definition,video_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_tags,video_duration_seconds,log_views
0,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,hd,56032799.0,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ...",231,17.841448
1,Singham Again | Official Trailer | A Rohit She...,Entertainment,hd,40832089.0,794000.0,623.0,False,"singham again ajay devgn,singam 3 movie,singha...",298,17.524979
2,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,hd,56032799.0,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ...",231,17.841448
3,Singham Again | Official Trailer | A Rohit She...,Entertainment,hd,40832089.0,794000.0,623.0,False,"singham again ajay devgn,singam 3 movie,singha...",298,17.524979
4,Bhool Bhulaiyaa 3 (Official Trailer): Kartik A...,Music,hd,56032799.0,276000000.0,21864.0,False,"tseries,tseries songs,bhool bhulaiyaa 3,bhool ...",231,17.841448


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# combining title + tags into one column
df_model["text"] = df_model["video_title"] + " " + df_model["video_tags"]
# this helps model understand content idea + keywords together

# TF-IDF vectorization (convert text → numbers)
tfidf = TfidfVectorizer(max_features=100)
# using top 100 important words to keep model efficient

text_features = tfidf.fit_transform(df_model["text"])

# converting to dataframe
text_df = pd.DataFrame(
    text_features.toarray(),
    columns=tfidf.get_feature_names_out()
)

# aligning index before merging
text_df.reset_index(drop=True, inplace=True)
df_model.reset_index(drop=True, inplace=True)

# combining text features with main dataframe
df_model = pd.concat([df_model, text_df], axis=1)

# encoding category + definition into numeric
df_model = pd.get_dummies(
    df_model,
    columns=["video_category_id", "video_definition"],
    drop_first=True
)
# ML model cannot understand text like "Music" → converts to 0/1 columns

# dropping columns we don't need anymore
df_model = df_model.drop(columns=[
    "video_title",
    "video_tags",
    "text"
])
# raw text removed because we already converted it

print("Dataset after encoding:", df_model.shape)
print("\nSome columns:")
print(df_model.columns[:20])

df_model.head()

Dataset after encoding: (302786, 120)

Some columns:
Index(['video_view_count', 'channel_subscriber_count', 'channel_video_count',
       'channel_have_hidden_subscribers', 'video_duration_seconds',
       'log_views', '2025', '2026', '3d', 'and', 'anirudh', 'bangla', 'best',
       'bgmi', 'bhajan', 'bhojpuri', 'bike', 'bollywood', 'brainrot',
       'challenge'],
      dtype='object')


,video_view_count,channel_subscriber_count,channel_video_count,channel_have_hidden_subscribers,video_duration_seconds,log_views,2025,2026,3d,and,...,video_category_id_Gaming,video_category_id_Howto & Style,video_category_id_Music,video_category_id_News & Politics,video_category_id_People & Blogs,video_category_id_Pets & Animals,video_category_id_Science & Technology,video_category_id_Sports,video_category_id_Travel & Events,video_definition_sd
0,56032799.0,276000000.0,21864.0,False,231,17.841448,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,False
1,40832089.0,794000.0,623.0,False,298,17.524979,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
2,56032799.0,276000000.0,21864.0,False,231,17.841448,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,False
3,40832089.0,794000.0,623.0,False,298,17.524979,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,False,False,False,False
4,56032799.0,276000000.0,21864.0,False,231,17.841448,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,False,False


### Why Random Forest?

- **Non-linear relationships** — even after log-transforming view counts, there is no simple linear relationship between features like duration, category, or keyword weights and the target. Linear Regression would underfit badly here.
- **Handles high-dimensional TF-IDF well** — TF-IDF produces 100 sparse columns (mostly zeros). Random Forest handles this naturally because each tree only looks at a random subset of features per split, effectively ignoring irrelevant keyword columns and reducing noise.
- **Parallel training** — trees are built independently (`n_jobs=-1`), making training fast on 300K rows with 120 features. Gradient Boosting and XGBoost build trees sequentially and would be significantly slower here.
- **Feature importance** — after training, we can check how much TF-IDF text features contribute vs structured features, which tells us whether the model is actually learning from content or just from channel metadata.
- **No feature scaling needed** — unlike SVR or KNN, Random Forest is not distance-based so raw TF-IDF weights and numeric columns can coexist without normalization.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# target (what we want to predict)
y = df_model["log_views"]
# using log scale because views are highly skewed

# features (input data)
X = df_model.drop(columns=["log_views", "video_view_count"])
# removing log target and original views to avoid leakage

# splitting into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# creating model
model_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# training model
model_reg.fit(X_train, y_train)

print("Regression model trained!")

Regression model trained!


In [4]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# predicting
y_pred = model_reg.predict(X_test)

# converting back to actual views
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

# calculating error
mae = mean_absolute_error(y_test_actual, y_pred_actual)

print("Average prediction error (views):", int(mae))

Average prediction error (views): 225533


My dataset ranges from:
few thousand views → millions → tens of millions
So 225K error is small relative to scale. So my model predicts good

In [5]:
# taking one sample from test set
sample = X_test.iloc[0:1]

# actual views
actual_views = int(np.expm1(y_test.iloc[0]))

# predicted views
predicted_views = int(np.expm1(model_reg.predict(sample)[0]))

print("Actual views:", actual_views)
print("Predicted views:", predicted_views)

Actual views: 9883032
Predicted views: 9909758


In [9]:
import joblib

# saving regression model
joblib.dump(model_reg, "../models/view_prediction_model.pkl")

# saving tfidf vectorizer
joblib.dump(tfidf, "../models/view_prediction_tfidf.pkl")

# saving training column names
joblib.dump(X.columns.tolist(), "../models/view_prediction_columns.pkl")

print("Regression model, tfidf, and columns saved!")

Regression model, tfidf, and columns saved!
